In [3]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt

In [4]:
X_train = np.load('../artifacts/X_train_preprocessed.npy')
X_test = np.load('../artifacts/X_test_preprocessed.npy')
y_train = np.load('../artifacts/y_train.npy')
y_test = np.load('../artifacts/y_test.npy')

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Доля оттока train: {y_train.mean():.2%}, test: {y_test.mean():.2%}")

Train: (5634, 38), Test: (1409, 38)
Доля оттока train: 26.54%, test: 26.54%


In [5]:
xgb_base = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42,
                         use_label_encoder=False, eval_metric='logloss')
xgb_base.fit(X_train, y_train)
y_proba_base = xgb_base.predict_proba(X_test)[:, 1]
print("Базовый XGBoost:")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_base):.4f}")
print(f"F1: {f1_score(y_test, xgb_base.predict(X_test)):.4f}")

Базовый XGBoost:
AUC-ROC: 0.8425
F1: 0.5808


In [6]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_proba_lr = lr.predict_proba(X_test)[:, 1]
print("Логистическая регрессия:")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_lr):.4f}")
print(f"F1: {f1_score(y_test, lr.predict(X_test)):.4f}")

Логистическая регрессия:
AUC-ROC: 0.8428
F1: 0.5872


In [7]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 9, 12],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5, 2.0]
}

xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(xgb, param_distributions=param_dist,
                                   n_iter=30, cv=cv, scoring='roc_auc',
                                   random_state=42, n_jobs=-1, verbose=1)
print("Запуск RandomizedSearchCV...")
random_search.fit(X_train, y_train)

print("\nЛучшие параметры:")
print(random_search.best_params_)
print(f"Лучший AUC-ROC на кросс-валидации: {random_search.best_score_:.4f}")

Запуск RandomizedSearchCV...
Fitting 3 folds for each of 30 candidates, totalling 90 fits

Лучшие параметры:
{'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Лучший AUC-ROC на кросс-валидации: 0.8496


In [8]:
best_xgb = random_search.best_estimator_
y_pred_best = best_xgb.predict(X_test)
y_proba_best = best_xgb.predict_proba(X_test)[:, 1]

print("Улучшенный XGBoost на тестовой выборке:")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_best):.4f}")
print(f"F1: {f1_score(y_test, y_pred_best):.4f}")
print(classification_report(y_test, y_pred_best))

Улучшенный XGBoost на тестовой выборке:
AUC-ROC: 0.8480
F1: 0.5658
              precision    recall  f1-score   support

           0       0.83      0.90      0.87      1035
           1       0.65      0.50      0.57       374

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [9]:
# Загрузим ранее сохранённую лучшую базовую модель (логистическая регрессия)
# или используем уже обученную lr

results = pd.DataFrame([
    {'Model': 'Logistic Regression',
     'AUC-ROC': roc_auc_score(y_test, y_proba_lr),
     'F1': f1_score(y_test, lr.predict(X_test))},
    {'Model': 'XGBoost (базовый)',
     'AUC-ROC': roc_auc_score(y_test, y_proba_base),
     'F1': f1_score(y_test, xgb_base.predict(X_test))},
    {'Model': 'XGBoost (улучшенный)',
     'AUC-ROC': roc_auc_score(y_test, y_proba_best),
     'F1': f1_score(y_test, y_pred_best)}
])

print("Сравнение моделей на тестовой выборке")
print(results.round(4))

Сравнение моделей на тестовой выборке
                  Model  AUC-ROC      F1
0   Logistic Regression   0.8428  0.5872
1     XGBoost (базовый)   0.8425  0.5808
2  XGBoost (улучшенный)   0.8480  0.5658


In [12]:
print("Улучшенный XGBoost превзошёл логистическую регрессию.")
joblib.dump(best_xgb, f'../artifacts/churn_model_final_XGBoost_improved.joblib')
print(f"Финальная модель сохранена как artifacts/churn_model_final_XGBoost_improved.joblib")
print(f"Лучшая модель: XGBoost_improved")
print(f"AUC-ROC финальной модели: {max(roc_auc_score(y_test, y_proba_best), roc_auc_score(y_test, y_proba_lr)):.4f}")

Улучшенный XGBoost превзошёл логистическую регрессию.
Финальная модель сохранена как artifacts/churn_model_final_XGBoost_improved.joblib
Лучшая модель: XGBoost_improved
AUC-ROC финальной модели: 0.8480
